# Boston Housing — Entrenamiento y Selección de Modelos

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

SQLITE_PATH = project_root / os.environ.get("SQLITE_PATH", ".storage/data.db")
TABLE_NAME = "housing_data"
MLFLOW_TRACKING_URI = "http://localhost:5001"

EXPERIMENT_NAME = "boston-housing-experiments"

os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("MINIO_ROOT_USER", "")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("MINIO_ROOT_PASSWORD", "")
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ.get("MINIO_ENDPOINT", "http://localhost:9000")


print(f"MLflow URI: {MLFLOW_TRACKING_URI}")
print(f"MinIO endpoint: {os.environ['MLFLOW_S3_ENDPOINT_URL']}")

MLflow URI: http://localhost:5001
MinIO endpoint: http://localhost:9000


In [2]:
# Dependencias necesarias
import sqlite3
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data.repository import load_data
from src.data.schema import FEATURE_COLUMNS, TARGET_COLUMN

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [3]:
# Carga inicial de datos
df = load_data(SQLITE_PATH, "housing_data")
print("Tamaño:", df.shape)
df.head()

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Tamaño: (506, 15)
Train: (455, 13), Test: (51, 13)


In [4]:
# Evaluar modelo
def evaluate(pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2": float(r2_score(y_test, y_pred)),
    }

# Construir preprocesador
def build_preprocessor():
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, FEATURE_COLUMNS)],
        remainder="drop",
        verbose_feature_names_out=False,
    )


In [5]:
# Experimento en MlFlow
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment: {EXPERIMENT_NAME}")

Experiment: boston-housing-experiments


In [6]:
# Entrenamiento con Ridge
ridge_params = {"alpha": 1.0, "random_state": 42}

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("estimator", Ridge(**ridge_params)),
])

with mlflow.start_run(run_name="ridge") as run:
    mlflow.set_tag("model_name", "ridge")
    mlflow.log_params(ridge_params)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    ridge_pipeline.fit(X_train, y_train)
    ridge_metrics = evaluate(ridge_pipeline, X_test, y_test)
    mlflow.log_metrics(ridge_metrics)

    mlflow.sklearn.log_model(
        sk_model=ridge_pipeline,
        artifact_path="model",
        input_example=X_train.iloc[:3],
    )

    ridge_run_id = run.info.run_id

print(f"Ridge run_id: {ridge_run_id}")
print(f"Ridge métricas: {ridge_metrics}")

2026/05/12 19:24:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 19:24:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/dalzoj/Documents/IA/boston-housing-mlops/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that inclu

🏃 View run ridge at: http://localhost:5001/#/experiments/1/runs/8596237d5b3e415cbe551ed7d4a78935
🧪 View experiment at: http://localhost:5001/#/experiments/1
Ridge run_id: 8596237d5b3e415cbe551ed7d4a78935
Ridge métricas: {'rmse': 3.777050087739426, 'mae': 2.701614821393705, 'r2': 0.771501783609689}


In [7]:
# Entrenamiento con Descenso del Gradiente

gb_params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.05,
    "random_state": 42,
}

gb_pipeline = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("estimator", GradientBoostingRegressor(**gb_params)),
])

with mlflow.start_run(run_name="gradient_boosting") as run:
    mlflow.set_tag("model_name", "gradient_boosting")
    mlflow.log_params(gb_params)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    gb_pipeline.fit(X_train, y_train)
    gb_metrics = evaluate(gb_pipeline, X_test, y_test)
    mlflow.log_metrics(gb_metrics)

    mlflow.sklearn.log_model(
        sk_model=gb_pipeline,
        artifact_path="model",
        input_example=X_train.iloc[:3],
    )

    gb_run_id = run.info.run_id

print(f"GBM run_id: {gb_run_id}")
print(f"GBM mpetricas: {gb_metrics}")

2026/05/12 19:25:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 19:25:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/dalzoj/Documents/IA/boston-housing-mlops/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that inclu

🏃 View run gradient_boosting at: http://localhost:5001/#/experiments/1/runs/05ec1152441e4ec38f417de1f2e7b644
🧪 View experiment at: http://localhost:5001/#/experiments/1
GBM run_id: 05ec1152441e4ec38f417de1f2e7b644
GBM mpetricas: {'rmse': 1.8860968283497308, 'mae': 1.5171286230821723, 'r2': 0.9430222499429509}


In [10]:
# Comparar mejor modelo
results = [
    {"model": "ridge", "run_id": ridge_run_id, **ridge_metrics},
    {"model": "gradient_boosting", "run_id": gb_run_id, **gb_metrics},
]

results_df = pd.DataFrame(results).set_index("model")
print(results_df.round(4))

winner = min(results, key=lambda r: r["rmse"])
print(f"Mejor modelo: {winner['model']} con RMSE={winner['rmse']:.4f}")

                                             run_id    rmse     mae      r2
model                                                                      
ridge              8596237d5b3e415cbe551ed7d4a78935  3.7771  2.7016  0.7715
gradient_boosting  05ec1152441e4ec38f417de1f2e7b644  1.8861  1.5171  0.9430
Mejor modelo: gradient_boosting con RMSE=1.8861


In [9]:
# Registrar modelo y promover a Staging

MODEL_REGISTRY_NAME = "boston-housing-regressor"

model_uri = f"runs:/{winner['run_id']}/model"
version = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)
print(f"Versión registrada: {version.version}")

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
client.transition_model_version_stage(
    name=MODEL_REGISTRY_NAME,
    version=version.version,
    stage="Staging",
    archive_existing_versions=False,
)
print(f"'{MODEL_REGISTRY_NAME}' v{version.version} ha sido promovido a Staging")

Successfully registered model 'boston-housing-regressor'.
2026/05/12 19:25:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 05ec1152441e4ec38f417de1f2e7b644 has no artifacts at artifact path 'model', registering model based on models:/m-f28ae9269a3d43428ca634b0add7b0fa instead
2026/05/12 19:25:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: boston-housing-regressor, version 1


Versión registrada: 1
'boston-housing-regressor' v1 ha sido promovido a Staging


Created version '1' of model 'boston-housing-regressor'.
/var/folders/wv/4s2g1dhx5yx7ybggv67jlt140000gn/T/ipykernel_85626/1456376524.py:10: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
